# Portfolio Risk & Optimization

Ce notebook constitue le pipeline reproductible du projet. Les prix proviennent directement de l'API `yfinance`; aucun fichier de marché n'est requis manuellement. La convention de risque est explicite : une perte est positive et vaut `-rendement`.

Les paramètres illustratifs sont documentés ici : période de 10 ans, 252 jours de bourse par an, taux sans risque annualisé de 2 %, portefeuille de référence équipondéré, 20 000 tirages Monte Carlo et contraintes long-only sans levier. Ces choix sont pédagogiques et doivent être adaptés au mandat réel d'un client.

In [ ]:
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yfinance as yf
from matplotlib.backends.backend_pdf import PdfPages
from scipy.optimize import minimize
from scipy.stats import norm

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='talk')
PROJECT_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'notebook').exists() and (path / 'README.md').exists())
PROCESSED_DIR = (PROJECT_ROOT / 'data' / 'processed').resolve()
FIGURES_DIR = (PROJECT_ROOT / 'reports' / 'figures').resolve()
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TICKERS = ['SPY', 'QQQ', 'TLT', 'GLD', 'IEF']
START_DATE = (pd.Timestamp.today() - pd.DateOffset(years=10)).strftime('%Y-%m-%d')
END_DATE = pd.Timestamp.today().strftime('%Y-%m-%d')
TRADING_DAYS = 252
RISK_FREE_RATE = 0.02
CONFIDENCE_LEVELS = [0.95, 0.99]
MC_SCENARIOS = 20_000
np.random.seed(42)
print(f'Projet: {PROJECT_ROOT}')
print(f'Fenêtre de données: {START_DATE} à {END_DATE}')

## 1. Data Collection (yfinance API)
Les ETF sont utilisés comme proxies liquides et diversifiés : actions américaines, actions technologiques, obligations d'État longues, or et obligations d'État intermédiaires.

In [ ]:
prices_raw = yf.download(TICKERS, start=START_DATE, end=END_DATE, auto_adjust=True, progress=False, threads=True)
if prices_raw.empty:
    raise RuntimeError('yfinance n a retourné aucune donnée. Vérifiez la connexion réseau ou réessayez.')
prices = prices_raw['Close'] if isinstance(prices_raw.columns, pd.MultiIndex) else prices_raw[['Close']]
prices = prices.reindex(columns=TICKERS).dropna(how='all').ffill().dropna()
returns = prices.pct_change().dropna()
if returns.shape[0] < 500:
    raise ValueError('Historique insuffisant pour une analyse robuste.')
returns.to_csv(PROCESSED_DIR / 'portfolio_returns.csv', index_label='Date')
print(f'{prices.shape[0]:,} observations de prix et {returns.shape[0]:,} rendements.')
prices.tail()

## 2. Exploratory Data Analysis
Les statistiques sont calculées sur les rendements quotidiens. L'annualisation utilise 252 séances et suppose une stabilité approximative des moments sur l'horizon analysé.

In [ ]:
annualized_return = (1 + returns.mean()) ** TRADING_DAYS - 1
annualized_volatility = returns.std() * np.sqrt(TRADING_DAYS)
summary = pd.DataFrame({'Rendement annualisé': annualized_return, 'Volatilité annualisée': annualized_volatility, 'Observations': returns.count()})
display(summary.style.format({'Rendement annualisé': '{:.2%}', 'Volatilité annualisée': '{:.2%}'}))
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
(prices / prices.iloc[0]).plot(ax=axes[0], title='Prix normalisés à 1')
returns.plot(kind='hist', bins=80, alpha=0.55, ax=axes[1], title='Distribution des rendements quotidiens')
axes[0].set_ylabel('Valeur normalisée'); axes[1].set_xlabel('Rendement')
plt.tight_layout(); plt.savefig(FIGURES_DIR / 'eda_prices_returns.png', dpi=160); plt.show()
plt.figure(figsize=(9, 7)); sns.heatmap(returns.corr(), annot=True, fmt='.2f', cmap='RdBu_r', center=0); plt.title('Corrélation des rendements'); plt.tight_layout(); plt.savefig(FIGURES_DIR / 'correlation_matrix.png', dpi=160); plt.show()

## 3-5. VaR historique, paramétrique et Monte Carlo
La VaR est exprimée comme une perte quotidienne positive. La méthode historique utilise le quantile empirique ; la méthode paramétrique suppose une loi normale ; Monte Carlo tire des rendements selon une normale multivariée calibrée sur les données. La comparaison doit être interprétée comme un diagnostic de modèle, pas comme une garantie.

In [ ]:
weights_equal = np.repeat(1 / len(TICKERS), len(TICKERS))
portfolio_returns = returns.to_numpy() @ weights_equal
portfolio_losses = -portfolio_returns
mean_daily = portfolio_returns.mean()
std_daily = portfolio_returns.std(ddof=1)
historical_var = {level: float(np.quantile(portfolio_losses, level)) for level in CONFIDENCE_LEVELS}
parametric_var = {level: float(-(mean_daily + std_daily * norm.ppf(1 - level))) for level in CONFIDENCE_LEVELS}
covariance = returns.cov().to_numpy()
mc_returns = np.random.multivariate_normal(returns.mean().to_numpy(), covariance, size=MC_SCENARIOS) @ weights_equal
mc_losses = -mc_returns
monte_carlo_var = {level: float(np.quantile(mc_losses, level)) for level in CONFIDENCE_LEVELS}
var_table = pd.DataFrame({'Historique': historical_var, 'Paramétrique': parametric_var, 'Monte Carlo': monte_carlo_var}).T
display(var_table.style.format('{:.2%}'))
assert all(value > 0 for value in historical_var.values()), 'La convention de perte attend une VaR positive.'
fig, ax = plt.subplots(figsize=(12, 5)); ax.hist(portfolio_losses, bins=100, alpha=0.7, density=True);
for level, color in zip(CONFIDENCE_LEVELS, ['#d95f02', '#7570b3']): ax.axvline(historical_var[level], color=color, label=f'Historique {level:.0%}')
ax.set_title('Distribution des pertes du portefeuille équipondéré'); ax.set_xlabel('Perte quotidienne'); ax.legend(); plt.tight_layout(); plt.savefig(FIGURES_DIR / 'var_distribution.png', dpi=160); plt.show()

## 6. Conditional VaR / Expected Shortfall
Le CVaR mesure la perte moyenne conditionnelle aux observations dépassant la VaR historique. Il est plus sensible à la queue de distribution et complète la VaR, qui ne décrit qu'un seuil.

In [ ]:
cvar = {level: float(portfolio_losses[portfolio_losses >= historical_var[level]].mean()) for level in CONFIDENCE_LEVELS}
risk_metrics = {
    'sample_start': str(returns.index.min().date()),
    'sample_end': str(returns.index.max().date()),
    'assets': TICKERS,
    'equal_weight': weights_equal.tolist(),
    'risk_free_rate': RISK_FREE_RATE,
    'annualized_return': float((1 + mean_daily) ** TRADING_DAYS - 1),
    'annualized_volatility': float(std_daily * np.sqrt(TRADING_DAYS)),
    'var_historical': {str(k): v for k, v in historical_var.items()},
    'var_parametric': {str(k): v for k, v in parametric_var.items()},
    'var_monte_carlo': {str(k): v for k, v in monte_carlo_var.items()},
    'cvar_historical': {str(k): v for k, v in cvar.items()}
}
with (PROCESSED_DIR / 'risk_metrics.json').open('w', encoding='utf-8') as file:
    json.dump(risk_metrics, file, indent=2)
pd.DataFrame({'VaR historique': historical_var, 'CVaR historique': cvar}).style.format('{:.2%}')

## 7. Optimisation de Markowitz
L'optimisation est long-only avec somme des poids égale à 1. La frontière cible des rendements compris entre le minimum et le maximum des rendements annualisés individuels. Le ratio de Sharpe utilise un taux sans risque de 2 % et ne constitue pas une prévision.

In [ ]:
mean_annual = returns.mean().to_numpy() * TRADING_DAYS
cov_annual = returns.cov().to_numpy() * TRADING_DAYS
asset_count = len(TICKERS)
bounds = tuple((0.0, 1.0) for _ in range(asset_count))
constraint = {'type': 'eq', 'fun': lambda weights: weights.sum() - 1}
def portfolio_stats(weights):
    annual_return = float(weights @ mean_annual)
    annual_volatility = float(np.sqrt(weights @ cov_annual @ weights))
    sharpe = (annual_return - RISK_FREE_RATE) / annual_volatility
    return annual_return, annual_volatility, float(sharpe)
def optimize(objective, initial=None):
    result = minimize(objective, initial or weights_equal, method='SLSQP', bounds=bounds, constraints=constraint, options={'maxiter': 1000, 'ftol': 1e-10})
    if not result.success: raise RuntimeError(result.message)
    return result.x
minimum_variance_weights = optimize(lambda w: portfolio_stats(w)[1])
maximum_sharpe_weights = optimize(lambda w: -portfolio_stats(w)[2])
target_returns = np.linspace(mean_annual.min(), mean_annual.max(), 40)
frontier_points = []
for target in target_returns:
    result = minimize(lambda w: portfolio_stats(w)[1], weights_equal, method='SLSQP', bounds=bounds, constraints=[constraint, {'type': 'eq', 'fun': lambda w, target=target: w @ mean_annual - target}], options={'maxiter': 1000})
    if result.success:
        annual_return, annual_volatility, sharpe = portfolio_stats(result.x); frontier_points.append({'return': annual_return, 'volatility': annual_volatility, 'sharpe': sharpe, 'weights': result.x.tolist()})
minimum_variance = portfolio_stats(minimum_variance_weights)
maximum_sharpe = portfolio_stats(maximum_sharpe_weights)
assert np.isclose(minimum_variance_weights.sum(), 1) and np.isclose(maximum_sharpe_weights.sum(), 1)
optimization_results = {'assets': TICKERS, 'frontier': {key: [point[key] for point in frontier_points] for key in ['return', 'volatility', 'sharpe']}, 'minimum_variance': {'weights': minimum_variance_weights.tolist(), 'return': minimum_variance[0], 'volatility': minimum_variance[1], 'sharpe': minimum_variance[2]}, 'maximum_sharpe': {'weights': maximum_sharpe_weights.tolist(), 'return': maximum_sharpe[0], 'volatility': maximum_sharpe[1], 'sharpe': maximum_sharpe[2]}}
with (PROCESSED_DIR / 'optimization_results.json').open('w', encoding='utf-8') as file: json.dump(optimization_results, file, indent=2)
frontier_frame = pd.DataFrame(optimization_results['frontier'])
fig, ax = plt.subplots(figsize=(10, 6)); ax.plot(frontier_frame['volatility'], frontier_frame['return'], label='Frontière efficiente')
ax.scatter(minimum_variance[1], minimum_variance[0], s=100, label='Variance minimale'); ax.scatter(maximum_sharpe[1], maximum_sharpe[0], s=100, label='Sharpe maximal'); ax.set(xlabel='Volatilité annualisée', ylabel='Rendement annualisé', title='Frontière efficiente'); ax.legend(); plt.tight_layout(); plt.savefig(FIGURES_DIR / 'efficient_frontier.png', dpi=160); plt.show()
print('Variance minimale:', minimum_variance); print('Sharpe maximal:', maximum_sharpe)

## 8. Stress Testing
Les fenêtres historiques sont des scénarios de marché observés, pas des prévisions. Le choc portefeuille est calculé comme la performance cumulée de l'allocation équipondérée pendant chaque période.

In [ ]:
stress_windows = {'Crise financière 2008': ('2008-09-01', '2009-03-31'), 'Krach COVID 2020': ('2020-02-19', '2020-03-23')}
stress_prices_raw = yf.download(TICKERS, start='2007-01-01', end=END_DATE, auto_adjust=True, progress=False, threads=True)['Close']
stress_prices = stress_prices_raw.reindex(columns=TICKERS).ffill().dropna()
stress_returns = stress_prices.pct_change().dropna()
stress_results = {}
for scenario, (start, end) in stress_windows.items():
    window = stress_returns.loc[start:end]
    if window.empty:
        raise ValueError(f'Aucune donnée disponible pour le scénario {scenario}.')
    stress_results[scenario] = {'start': start, 'end': end, 'observations': int(len(window)), 'portfolio_return': float((1 + window.to_numpy() @ weights_equal).prod() - 1)}
stress_table = pd.DataFrame(stress_results).T
display(stress_table.style.format({'portfolio_return': '{:.2%}'}))
with (PROCESSED_DIR / 'stress_results.json').open('w', encoding='utf-8') as file: json.dump(stress_results, file, indent=2)

## 9. Conclusions & Business Insights
La diversification doit être évaluée à partir des corrélations et de la contribution au risque, pas seulement du nombre de lignes. La VaR paramétrique peut sous-estimer les queues épaisses ; le CVaR et le stress testing rendent cette limite visible. Markowitz est sensible aux estimations de rendement et de covariance : les poids optimaux doivent donc être présentés comme une aide à la décision, avec contraintes et validation hors échantillon.

Le dashboard Streamlit lit les artefacts JSON/CSV générés ci-dessus. Le PDF ci-dessous est généré à partir des mêmes résultats et figures, sans chiffres saisis manuellement.

In [ ]:
report_path = PROJECT_ROOT / 'reports' / 'final_report.pdf'
with PdfPages(report_path) as pdf:
    fig = plt.figure(figsize=(8.27, 11.69)); fig.text(0.08, 0.88, 'Portfolio Risk & Optimization', fontsize=25, weight='bold'); fig.text(0.08, 0.83, 'Rapport exécutif généré depuis les données yfinance', fontsize=13)
    text = (
        f"Période: {risk_metrics['sample_start']} à {risk_metrics['sample_end']}\n\n"
        f"Rendement annualisé du portefeuille équipondéré: {risk_metrics['annualized_return']:.2%}\n"
        f"Volatilité annualisée: {risk_metrics['annualized_volatility']:.2%}\n"
        f"VaR historique 95%: {risk_metrics['var_historical']['0.95']:.2%}\n"
        f"CVaR historique 95%: {risk_metrics['cvar_historical']['0.95']:.2%}\n\n"
        "Conclusion: les trois modèles de VaR donnent des ordres de grandeur comparables "
        "mais diffèrent dans la queue de distribution. Le CVaR et les scénarios historiques "
        "complètent la mesure ponctuelle de VaR. L'optimisation est sensible aux hypothèses "
        "et doit être revue régulièrement."
    )
    fig.text(0.08, 0.70, text, fontsize=12, va='top', linespacing=1.7); fig.text(0.08, 0.08, 'Document pédagogique - aucune recommandation personnalisée', fontsize=9); pdf.savefig(fig); plt.close(fig)
    for figure_path in ['eda_prices_returns.png', 'correlation_matrix.png', 'var_distribution.png', 'efficient_frontier.png']:
        image = plt.imread(FIGURES_DIR / figure_path); fig, ax = plt.subplots(figsize=(8.27, 11.69)); ax.imshow(image); ax.axis('off'); ax.set_title(figure_path.replace('_', ' ').replace('.png', '').title()); pdf.savefig(fig, bbox_inches='tight'); plt.close(fig)
    fig, ax = plt.subplots(figsize=(8.27, 11.69)); ax.axis('off'); ax.text(0.05, 0.92, 'Méthodologie, limites et recommandations', fontsize=18, weight='bold'); ax.text(0.05, 0.82, 'Méthodologie\n- Rendements quotidiens téléchargés via yfinance.\n- VaR historique, normale et Monte Carlo multivariée.\n- CVaR comme moyenne des pertes au-delà de la VaR.\n- Optimisation long-only sous contrainte de somme des poids.\n- Stress tests sur deux fenêtres de crise observées.\n\nLimites\n- Les rendements passés ne préjugent pas des rendements futurs.\n- La normale ne capture pas parfaitement les queues épaisses.\n- Markowitz est instable lorsque les paramètres sont estimés sur un historique court.\n- Les ETF sont des proxies, non des obligations de performance.\n\nActions recommandées\n1. Ajouter une validation walk-forward et des coûts de transaction.\n2. Tester des contraintes de poids et une optimisation robuste.\n3. Contrôler les résultats et recalibrer les données selon le mandat.', fontsize=12, va='top', linespacing=1.6); pdf.savefig(fig); plt.close(fig)
print(f'Rapport généré: {report_path}')